# 02. 공영 물품보관함 실시간 API → `lockers.json`

이 노트북은 공공데이터포털에서 승인받은 **「행정안전부 한국지역정보개발원_(전국 통합데이터) 공영 물품보관함 현황 실시간 정보」** API를 불러서,
올림픽공원 주변 보관함만 골라 웹 데모가 읽는 `docs/data/lockers.json` 파일로 바꿔 줍니다.

| 단계 | 하는 일 |
|---|---|
| 0 | 설정 (중심 위치, 반경) |
| 1 | 인증키 입력 (화면에 안 보이고, 파일에도 저장 안 됨) |
| 2 | API 3개 부르기: 위치 정보, 상세 정보(요금), 실시간 빈 칸 |
| 3 | 올림픽공원 반경 안 보관함만 고르기 |
| 4 | `lockers.json` 만들고 내려받기 → GitHub `docs/data/` 에 업로드 |

> **왜 웹 화면에서 바로 API를 부르지 않나요?**
> GitHub Pages 는 누구나 소스를 볼 수 있어서, 화면 코드에 인증키를 넣으면 키가 공개됩니다.
> 그래서 키는 Colab 에서만 쓰고, 결과(JSON)만 올립니다. 발표 직전에 이 노트북을 한 번 더 돌리면 최신 값이 됩니다.

**인증키를 GitHub, 노션, 단톡방에 붙여 넣지 마세요.** 이 노트북은 실행할 때마다 키를 물어봅니다.

## 0. 설정

In [ ]:
# 올림픽공원 중심 (평화의광장 부근) 과 찾을 반경
CENTER_LAT, CENTER_LON = 37.5207, 127.1214
RADIUS_M = 1500          # 1.5km 안 보관함만 사용. 너무 적게 나오면 2000 으로 늘려 보기

# 공연장 좌표 (보관함마다 "가까운 공연장" 을 정할 때 사용, 600m 이내면 가깝다고 봄)
VENUES = {
    "dome":  (37.5196, 127.1273),   # KSPO DOME
    "hall":  (37.5180, 127.1250),   # 올림픽홀
    "arena": (37.5186, 127.1300),   # 핸드볼경기장
    "lawn":  (37.5213, 127.1196),   # 88잔디마당
}
NEAR_M = 600

API_BASE = "https://apis.data.go.kr/B551982/psl_v2"
# 지역 코드(법정동 시도 코드). 서울특별시 = 1100000000. 데이터가 안 나오면(K3) 이 값을 넣고/빼고 자동으로 다시 시도함
STDG_CD = "1100000000"

## 1. 인증키 입력
공공데이터포털 **마이페이지 → 데이터 활용 → Open API → 활용신청 현황 → 이 API** 에 있는 **일반 인증키(Decoding)** 를 붙여 넣고 Enter.
(Encoding 키를 넣어도 아래 코드가 알아서 처리합니다)

In [ ]:
from getpass import getpass
from urllib.parse import unquote
SERVICE_KEY = unquote(getpass("인증키를 붙여 넣고 Enter: ").strip())
print("키 길이:", len(SERVICE_KEY), "(화면에는 키를 보여 주지 않습니다)")

## 2. API 3개 부르기
- `locker_info_v2` : 보관함 위치, 이름, 운영 시간
- `locker_detail_info_v2` : 크기, 요금 설명
- `locker_realtime_use_v2` : **크기별 지금 빈 칸 수** (몇 분마다 갱신)

결과 코드가 `K0` 이면 정상입니다. 오류가 나면 아래 표를 보세요.

| 보이는 오류 | 뜻 | 할 일 |
|---|---|---|
| `SERVICE_KEY_IS_NOT_REGISTERED` | 키가 아직 등록 전이거나 잘못 붙여 넣음 | 승인 후 1~2시간 기다리기, Decoding/Encoding 키 바꿔 보기 |
| `K3` `NODATA_ERROR` | 키는 정상, 조건에 맞는 데이터가 없음 | 아래 코드가 지역 코드(`stdgCd`)를 넣고/빼고 자동 재시도. 그래도 없으면 일시적 공백이라 잠시 뒤 다시 |
| `K10` 등 K로 시작하는 다른 코드 | 요청 값 문제 | `numOfRows` 를 1000 이하로 |
| `LIMITED_NUMBER_OF_SERVICE_REQUESTS_EXCEEDS` | 하루 호출 횟수 초과 | 내일 다시 |

In [ ]:
import requests, json

def call_once(path, extra, rows=1000):
    # API 한 개를 불러 (결과 코드, 메시지, item 목록) 을 돌려줌. 여러 쪽이면 끝까지 넘김
    items, page = [], 1
    while True:
        r = requests.get(f"{API_BASE}/{path}", timeout=20, params={
            "serviceKey": SERVICE_KEY, "type": "JSON", "pageNo": page, "numOfRows": rows, **extra})
        try:
            body = r.json()
        except ValueError:
            raise RuntimeError(f"{path}: JSON 이 아닌 응답 (HTTP {r.status_code})\n{r.text[:300]}")
        head = body.get("header", {})
        code = head.get("resultCode")
        if code != "K0":
            return code, head.get("resultMsg"), items
        got = body.get("body", {}).get("item") or []
        got = got if isinstance(got, list) else [got]
        items += got
        total = int(body.get("body", {}).get("totalCount") or len(items))
        if len(items) >= total or not got:
            return code, "", items
        page += 1

def call(path):
    # 지역 코드 없이, 그다음 지역 코드 넣어서 순서로 시도. K3(데이터 없음)이면 다음 방법으로
    tries = [{}] + ([{"stdgCd": STDG_CD}] if STDG_CD else [])
    for extra in tries:
        code, msg, items = call_once(path, extra)
        print(f"  {path} {extra or '(지역 코드 없음)'} → {code} {msg} {len(items)}개")
        if code == "K0" and items:
            return items
        if code not in ("K0", "K3"):
            raise RuntimeError(f"{path}: 결과 코드 {code} {msg}")
    return []

info     = call("locker_info_v2")
detail   = call("locker_detail_info_v2")
realtime = call("locker_realtime_use_v2")
print(f"위치 {len(info)}개, 상세 {len(detail)}개, 실시간 {len(realtime)}개")
if not info or not realtime:
    raise RuntimeError("위치 또는 실시간 데이터가 비어 있습니다(K3). 잠시 뒤 다시 실행하거나, 위 출력 줄을 캡처해서 공유해 주세요.")
print("첫 번째 위치 예시:", json.dumps(info[0], ensure_ascii=False)[:400])

## 3. 올림픽공원 주변 보관함만 고르기

In [ ]:
import math, re

def dist_m(a_lat, a_lon, b_lat, b_lon):
    R = 6371000
    p1, p2 = math.radians(a_lat), math.radians(b_lat)
    dp, dl = p2 - p1, math.radians(b_lon - a_lon)
    h = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(h))

def num(v):
    try: return float(v)
    except (TypeError, ValueError): return None

rt_by_id = {str(r.get("stlckId")): r for r in realtime}
dt_by_id = {str(r.get("stlckId")): r for r in detail}

nearby = []
for r in info:
    lat, lon = num(r.get("lat")), num(r.get("lot"))
    if lat is None or lon is None:
        continue
    d = dist_m(CENTER_LAT, CENTER_LON, lat, lon)
    if d <= RADIUS_M:
        nearby.append((round(d), r))
nearby.sort(key=lambda x: x[0])

print(f"반경 {RADIUS_M}m 안 보관함: {len(nearby)}개")
for d, r in nearby:
    rt = rt_by_id.get(str(r.get("stlckId")), {})
    print(f"  {d:>5}m  {r.get('stlckRprsPstnNm')}  |  {r.get('stlckDtlPstnNm')}  |  "
          f"빈 칸 대 {rt.get('usePsbltyLrgszStlckCnt','?')} 중 {rt.get('usePsbltyMdmszStlckCnt','?')} 소 {rt.get('usePsbltySmlszStlckCnt','?')}")
if not nearby:
    print("주변 보관함이 없습니다. RADIUS_M 을 늘리거나, 위 '첫 번째 위치 예시' 의 lat, lot 값을 확인하세요.")

## 4. `lockers.json` 만들기
웹 화면 형식에 맞춰 바꿉니다.
- 이 API 는 **빈 칸 수만** 주고 크기별 전체 칸 수는 주지 않습니다. 그래서 `total` 은 비워 두고, 화면은 빈 칸 수로 상태를 표시합니다 (0칸 만석, 3칸 이하 얼마 안 남음).
- 데모 지도는 개략도라서, 지도 위치(x, y)는 아는 역은 역 표시 옆에 고정하고 나머지는 대략 배치합니다.

In [ ]:
from datetime import datetime

def hhmm(v):
    v = str(v or "")
    return f"{v[:2]}:{v[2:4]}" if len(v) >= 4 else ""

# 데모 지도는 실제 축척이 아닌 개략도라서, 아는 역은 지도 위 역 표시 옆에 고정하고 나머지는 경도로 대략 배치
KNOWN = {  # 이름에 들어 있는 글자: (x, y, 영어 이름)
    "올림픽공원": (318, 160, "Olympic Park Stn"),
    "몽촌토성":   (58, 124, "Mongchontoseong Stn"),
    "한성백제":   (126, 270, "Hanseong Baekje Stn"),
}
def place_of(name, lat, lon):
    for k, (x, y, en) in KNOWN.items():
        if k in name:
            line = re.match(r"^(\d)호선", name)
            return x, y, en + (f" (Line {line.group(1)})" if line else "")
    x = 30 + (lon - 127.1123) / 0.0185 * 304      # 몽촌토성역(30) ~ 올림픽공원역(334) 사이 비례
    return round(min(max(x, 14), 346)), 150, name

def cnt(v):
    try: return int(float(v))
    except (TypeError, ValueError): return None

lockers, latest = [], ""
for d, r in nearby:
    sid = str(r.get("stlckId"))
    rt = rt_by_id.get(sid)
    if not rt:
        continue                      # 실시간 값이 없는 보관함은 '만석' 으로 오해할 수 있어 뺌
    lat, lon = float(r["lat"]), float(r["lot"])
    name = str(r.get("stlckRprsPstnNm") or sid)
    x, y, name_en = place_of(name, lat, lon)
    sizes = {}
    for key, field in (("S", "usePsbltySmlszStlckCnt"), ("M", "usePsbltyMdmszStlckCnt"), ("L", "usePsbltyLrgszStlckCnt")):
        e = cnt(rt.get(field))
        if e is not None:
            sizes[key] = {"total": None, "empty": e}
    if not sizes:
        continue
    t = str(rt.get("totDt") or "")
    latest = max(latest, t)
    spot = str(r.get("stlckDtlPstnNm") or "")
    open_, close = hhmm(r.get("wkdyOperBgngTm")), hhmm(r.get("wkdyOperEndTm"))
    lockers.append({
        "id": sid, "x": x, "y": y, "lat": lat, "lon": lon, "dist_m": d,
        "near": [v for v, (vl, vo) in VENUES.items() if dist_m(lat, lon, vl, vo) <= NEAR_M],
        "op": True,
        "hours": f"{open_}~{close}" if open_ else "",
        "name": {"ko": name, "en": name_en},   # 영어 이름은 API 에 없어 KNOWN 표로 만듦
        "spot": {"ko": spot, "en": spot},
        "fee": str(dt_by_id.get(sid, {}).get("utztnCrgExpln") or ""),
        "sizes": sizes,
    })

updated = f"{latest[8:10]}:{latest[10:12]}" if len(latest) >= 12 else datetime.now().strftime("%H:%M")
out = {
    "_설명": "행정안전부 한국지역정보개발원 공영 물품보관함 실시간 API 결과를 02_lockers_api 노트북으로 변환. total 은 API 에 없어 비움.",
    "sample": False,
    "source": "행정안전부 한국지역정보개발원_(전국 통합데이터) 공영 물품보관함 현황 실시간 정보 (공공데이터포털)",
    "fetched_at": (latest[:8] + " " + updated) if latest else datetime.now().strftime("%Y%m%d %H:%M"),
    "updated": updated,
    "lockers": lockers,
}
with open("lockers.json", "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=1)
print(f"보관함 {len(lockers)}개를 lockers.json 에 저장했습니다. 기준 시각 {updated}")

In [ ]:
# 내려받기 (Colab 에서만 동작). 받은 파일을 GitHub docs/data/ 에 올리면 화면이 바뀝니다.
try:
    from google.colab import files
    files.download("lockers.json")
except ImportError:
    print("Colab 이 아니라서 현재 폴더에 lockers.json 만 저장했습니다.")